### **Imports**

In [7]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

### **Clean Data Ingestion**

In [3]:
# 1. Load raw data
df_raw = pd.read_csv('../data/data.csv')

# 2. Apply cleaning rules established in Notebook 01
df = df_raw.copy()

# Filter zero prices and extreme high-end outliers (> $5M)
df = df[(df['price'] > 0) & (df['price'] < 5_000_000)]

# Filter invalid room counts
df = df[~((df['bedrooms'] == 0) & (df['bathrooms'] == 0))]

# Parse date and add log target
df['date'] = pd.to_datetime(df['date'])
df['log_price'] = np.log1p(df['price'])

print(f"Dataset successfully loaded & cleaned: {df.shape[0]} rows, {df.shape[1]} columns")

Dataset successfully loaded & cleaned: 4546 rows, 19 columns


### **Age & Time-Based Features**

Calculate how old the house was when sold, whether it was ever renovated, and extract seasonal time trends from the sales date.

In [4]:
# Extract transaction year from date
df['sale_year'] = df['date'].dt.year
df['sale_month'] = df['date'].dt.month

# 1. House Age at time of sale
df['house_age'] = df['sale_year'] - df['yr_built']

# 2. Renovation status & years since renovation
df['is_renovated'] = (df['yr_renovated'] > 0).astype(int)

# If renovated, calculate years since renovation; otherwise, set to house_age
df['years_since_renovation'] = np.where(
    df['is_renovated'] == 1,
    df['sale_year'] - df['yr_renovated'],
    df['house_age']
)

# Sanity check: Ensure no negative age values
df['house_age'] = df['house_age'].clip(lower=0)
df['years_since_renovation'] = df['years_since_renovation'].clip(lower=0)

df[['yr_built', 'yr_renovated', 'house_age', 'is_renovated', 'years_since_renovation']].head()

,yr_built,yr_renovated,house_age,is_renovated,years_since_renovation
0,1955,2005,59,1,9
1,1921,0,93,0,93
2,1966,0,48,0,48
3,1963,0,51,0,51
4,1976,1992,38,1,22


### **Room & Living Area Ratios**

In [5]:
# 1. Total rooms and bed-to-bath ratio
df['total_rooms'] = df['bedrooms'] + df['bathrooms']
df['bed_bath_ratio'] = df['bedrooms'] / (df['bathrooms'] + 0.001)  # Avoid division by zero

# 2. Average room size (sqft per room)
df['sqft_per_room'] = df['sqft_living'] / (df['total_rooms'] + 0.001)

# 3. Lot utilization ratio (living area vs total lot)
df['living_to_lot_ratio'] = df['sqft_living'] / df['sqft_lot']

# 4. Basement ratio (basement area relative to total living space)
df['basement_ratio'] = df['sqft_basement'] / df['sqft_living']
df['has_basement'] = (df['sqft_basement'] > 0).astype(int)

df[['total_rooms', 'bed_bath_ratio', 'sqft_per_room', 'living_to_lot_ratio', 'has_basement']].head()

,total_rooms,bed_bath_ratio,sqft_per_room,living_to_lot_ratio,has_basement
0,4.50,1.998668,297.711620,0.169363,0
1,7.50,1.999200,486.601786,0.403315,1
2,5.00,1.499250,385.922815,0.161547,0
3,5.25,1.332741,380.879832,0.249066,1
4,6.50,1.599360,298.415628,0.184762,1


### **Location Grouping (Frequency & Target Encoding)**

Instead of using hundreds of one-hot encoded binary columns for statezip or city, create aggregated location features based on historical median prices.

In [6]:
# 1. Median price per city
city_price_map = df.groupby('city')['price'].median().to_dict()
df['city_median_price'] = df['city'].map(city_price_map)

# 2. Median price per statezip
zip_price_map = df.groupby('statezip')['price'].median().to_dict()
df['zip_median_price'] = df['statezip'].map(zip_price_map)

# 3. Zip code density / property count
zip_count_map = df.groupby('statezip')['statezip'].count().to_dict()
df['zip_property_count'] = df['statezip'].map(zip_count_map)

df[['city', 'city_median_price', 'statezip', 'zip_median_price', 'zip_property_count']].head()

,city,city_median_price,statezip,zip_median_price,zip_property_count
0,Shoreline,360000.0,WA 98133,371500.0,93
1,Seattle,490000.0,WA 98119,780000.0,49
2,Kent,283200.0,WA 98042,290500.0,98
3,Bellevue,725000.0,WA 98008,526500.0,48
4,Redmond,636500.0,WA 98052,605000.0,135


### **Save Processed Dataset to Disk**

In [8]:
# Create directory if it doesn't exist
os.makedirs('../data/processed', exist_ok=True)

# Save cleaned and feature-engineered dataset
output_path = '../data/processed/cleaned_housing_data.csv'
df.to_csv(output_path, index=False)

print(f"✓ Feature engineering complete! Dataset saved to: {output_path}")
print(f"Final shape: {df.shape[0]} rows, {df.shape[1]} columns")

✓ Feature engineering complete! Dataset saved to: ../data/processed/cleaned_housing_data.csv
Final shape: 4546 rows, 33 columns
